# 05.1 — The custom training loop

MATLAB's `trainNetwork` hides the training loop. This project — like most serious PyTorch code — writes the loop **by hand**, because a custom pipeline needs control MATLAB's black box doesn't give: per-component losses, gradient accumulation, curriculum schedules, checkpointing on its own terms. This notebook is the foundation of Module 05: the loop itself, from first principles to the production `fit_supervised`.

This notebook covers:

* The five-line core loop, and the MATLAB `trainNetwork` it replaces.
* Epoch vs iteration; the train/validate rhythm.
* `model.train()` / `model.eval()` + `torch.no_grad()` around validation.
* How the production loop layers features onto this same skeleton.

**Prerequisites:** [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [02.7 optimizers](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb), [04.6 classifier](../04_architecture/04.6_multi_head_classifier.ipynb).

## Section 1 — What MATLAB does

MATLAB offers two levels. The high-level one hides everything:

```matlab
net = trainNetwork(data, layers, trainingOptions('adam', ...));
```

The pipeline uses the **custom** loop (`cgg_trainNetwork`) because it needs to reach inside:

```matlab
for epoch = 1:NumEpochs
    for batch = 1:NumBatches
        [X, T] = nextBatch(...);
        [loss, grads] = dlfeval(@modelGradients, net, X, T);
        [net, avgG, avgSqG] = adamupdate(net, grads, avgG, avgSqG, iter, lr);
    end
    validateAndCheckpoint(net);
end
```

Every mechanism you'll meet in Module 05 — accumulation, clipping, scheduling, checkpointing, curricula — lives *inside* that double loop. PyTorch's version is the same shape, and once you can write the five-line core, the rest is layers on top.

## Section 2 — The Python concepts you need

### 2.1 — The core loop, minimal

Strip everything away and a supervised training loop is five lines per batch — the rhythm from [02.7 §2.1](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb), now inside an epoch loop:

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 3))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

# Toy data: 200 examples, learnable structure
X = torch.randn(200, 8)
y = (X @ torch.randn(8, 3)).argmax(1)

def batches(X, y, bs):
    for i in range(0, len(X), bs):
        yield X[i:i+bs], y[i:i+bs]

for epoch in range(10):                         # ── epoch loop
    for xb, yb in batches(X, y, bs=32):         # ── iteration loop
        optimizer.zero_grad()                   # 1. clear gradients
        loss = loss_fn(model(xb), yb)           # 2. forward + loss
        loss.backward()                         # 3. backward
        optimizer.step()                        # 4. update
    if epoch % 3 == 0:
        acc = (model(X).argmax(1) == y).float().mean()
        print(f"epoch {epoch}: loss {loss.item():.3f}, acc {acc:.2f}")

That's the whole idea. **Epoch** = one pass over all data; **iteration** = one batch. The four numbered lines are the MATLAB `dlfeval` + `adamupdate` (which [02.7 §2.1](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb) mapped). Everything Module 05 adds — accumulation splits step 3, clipping sits between 3 and 4, schedulers fire after the inner loop — hangs off this skeleton.

### 2.2 — Train vs validate: mode and no_grad

A real loop *validates* after each epoch to track generalization. Validation must (a) put the model in `eval()` mode so dropout/batch-norm behave correctly, and (b) run under `torch.no_grad()` so no graph is built ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [02.6 §2.5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)):

In [ ]:
def validate(model, X, y):
    model.eval()                                # dropout off, batch-norm frozen
    with torch.no_grad():                       # no graph, ~half the memory, faster
        logits = model(X)
        loss = nn.functional.cross_entropy(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
    model.train()                               # BACK to train mode for the next epoch
    return loss.item(), acc.item()

Xv = torch.randn(60, 8); yv = (Xv @ torch.randn(8, 3)).argmax(1)
print("validation:", validate(model, Xv, yv))

The `eval()` → validate → `train()` dance is mandatory and easy to forget. Skip `eval()` and dropout corrupts your metrics ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)); skip the restore to `train()` and the *next* epoch trains with dropout off. The production loop does both around every validation pass.

### 2.3 — The full skeleton, assembled

In [ ]:
def train(model, opt, loss_fn, X, y, Xv, yv, epochs):
    history = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in batches(X, y, bs=32):
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        val_loss, val_acc = validate(model, Xv, yv)   # eval mode inside
        history.append({"epoch": epoch, "val_loss": val_loss, "val_acc": val_acc})
    return history

torch.manual_seed(0)
m = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 3))
hist = train(m, torch.optim.Adam(m.parameters(), lr=1e-2), nn.CrossEntropyLoss(),
             X, y, Xv, yv, epochs=15)
print(f"first val_acc {hist[0]['val_acc']:.2f} → last val_acc {hist[-1]['val_acc']:.2f}")

This `train(...)` is `fit_supervised` in miniature: an epoch loop, per-batch update, per-epoch validation, and a returned history. The production version returns a list of `EpochHistory` objects and layers on everything else — but the bones are these.

## Section 3 — The `neural_data_decoding` implementation

`train_one_epoch`'s core — the same four steps, wrapped in the project's per-component-loss machinery:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/loop.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if ".backward()" in line)
for k in range(max(0, i - 8), min(i + 6, len(src))):
    marker = " ►" if any(t in src[k] for t in ("zero_grad", ".backward()", ".step()")) else "  "
    print(f"{k + 1:4}{marker}| {src[k]}")

Things to spot:

* `optimizer.zero_grad()` → build `total_loss` → `total_loss.backward()` → (clip) → `optimizer.step()` — §2.1's four steps, unmistakable under the extra bookkeeping.
* One `total_loss.backward()` even though the loss sums reconstruction + KL + classification + confidence ([02.5 §3](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), Critical Note #28) — the graph distributes each component's gradient automatically.
* The function returns metrics + updated loss priors + confidence history — richer than a scalar loss, because the pipeline tracks per-component state across epochs.

And the epoch-level orchestration — `fit_supervised`'s signature tells you what the skeleton grew into:

In [ ]:
life = Path("../../src/neural_data_decoding/training/lifecycle.py").read_text().splitlines()
i = next(j for j, line in enumerate(life) if line.startswith("def fit_supervised"))
# print the signature block up to the closing ) -> ...
k = i
while k < len(life) and not life[k].rstrip().endswith("EpochHistory]:"):
    print(f"{k + 1:4} | {life[k]}")
    k += 1
print(f"{k + 1:4} | {life[k]}")

Every keyword argument is one Module 05 topic: `grad_clip_norm` ([05.3](05.3_gradient_clipping.ipynb)), `accumulation_max_size` ([05.2](05.2_gradient_accumulation.ipynb)), `curriculum` ([Module 07](../README.md)), `checkpoint_dir` ([05.5](05.5_checkpoint_resume_state_machine.ipynb)), `loss_priors`/`confidence_history` ([Module 06](../README.md)). The loop is the trunk; these are the branches.

## Section 4 — Hands-on exercises

### Exercise 4.1 — spot the missing step

The loop below trains but the loss never decreases. One of the four steps is missing — which, and what's the symptom?

In [ ]:
torch.manual_seed(0)
m = nn.Sequential(nn.Linear(8, 3))
opt = torch.optim.Adam(m.parameters(), lr=1e-1)
losses = []
for epoch in range(5):
    for xb, yb in batches(X, y, bs=32):
        loss = nn.functional.cross_entropy(m(xb), yb)
        loss.backward()
        opt.step()
        # (something is missing here)
    losses.append(loss.item())
print("losses:", [round(l, 2) for l in losses])

In [ ]:
# Reveal: opt.zero_grad() is missing, so gradients ACCUMULATE across every
# iteration (02.5 §2.3) — each step applies the sum of all past gradients,
# and training behaves erratically. The fix:
torch.manual_seed(0)
m = nn.Sequential(nn.Linear(8, 3))
opt = torch.optim.Adam(m.parameters(), lr=1e-1)
losses = []
for epoch in range(5):
    for xb, yb in batches(X, y, bs=32):
        opt.zero_grad()                     # ← the missing step
        loss = nn.functional.cross_entropy(m(xb), yb)
        loss.backward()
        opt.step()
    losses.append(loss.item())
print("fixed losses:", [round(l, 2) for l in losses])

### Exercise 4.2 — the eval() bug

Write a validation function that FORGETS to restore `model.train()`. Then show that the epoch after it trains with dropout disabled (build a model with dropout, check `model.training` before and after).

In [ ]:
# Reveal:
m = nn.Sequential(nn.Linear(8, 8), nn.Dropout(0.5), nn.Linear(8, 3))
def bad_validate(model):
    model.eval()
    with torch.no_grad():
        _ = model(X)
    # forgot model.train()!
print("before validate, training mode?", m.training)
bad_validate(m)
print("after  validate, training mode?", m.training, "← dropout now OFF for next epoch (bug)")

## Section 5 — Common errors

### Loss doesn't decrease / trains erratically

A missing or misplaced `zero_grad()` (Exercise 4.1) — gradients accumulate across iterations. The order is `zero_grad → forward → backward → step`, every iteration.

### Validation metrics noisier / worse than training

Missing `model.eval()` — dropout/batch-norm active during validation ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)). And its partner: wrap validation in `torch.no_grad()` or you build (and leak) a graph you never use.

### The epoch after validation trains wrong

Forgot to restore `model.train()` after `eval()` (Exercise 4.2). Always flip back.

### Out of memory during validation

No `torch.no_grad()` — the graph is being retained for a backward that never comes. Validation is inference; no graph needed.

### Training works but is mysteriously slow

Often `.item()` / `.cpu()` calls inside the inner loop force a device sync every iteration ([02.4 §5](../02_numpy_and_pytorch_basics/02.4_pytorch_tensors_intro.ipynb)). Accumulate on-device, sync once per epoch.

## Section 6 — Further reading

- [PyTorch training loop tutorial](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) — the canonical hand-written loop.
- [`src/neural_data_decoding/training/loop.py`](../../src/neural_data_decoding/training/loop.py) — `train_one_epoch` + `validate`.
- [`src/neural_data_decoding/training/lifecycle.py`](../../src/neural_data_decoding/training/lifecycle.py) — `fit_supervised`, the epoch orchestrator.

Next up: **[05.2 gradient accumulation](05.2_gradient_accumulation.ipynb)** — training with batches too big for the GPU, by splitting step 3.